# news_company_relation — 종목과 상대 기업의 관계 1문장

뉴스 상세 화면에서 "이 기사가 왜 내 종목과 이어졌는지"를 알려 주는 **관계 문장**을 만든다(2026-11-20).
예: "SK하이닉스는 엔비디아에 AI 가속기용 메모리를 공급해요.", "셀트리온은 일라이 릴리의 의약품을 위탁생산하는 회사예요."

```
news.term_links(서버 DB) ─ 종목–검색어마다 그래프 관계(관계 이름·방향·내용·금액·근거 문장)
   ▼
news_company_relation.ipynb ─ 해외 상대(기업)와 직접 관계(link = direct)인 짝만 → (종목, 상대 기업)마다 하나로 합침
                               → 이미 만든 짝(관계가 같으면) 건너뜀 → OpenAI(BATCH_SIZE짝씩 한 번 호출)
   ▼  company_relations.jsonl (짝마다 한 줄)
```

- 관계 문장은 **기사와 상관없이 짝마다 같다.** 그래서 기사마다 만들지 않고 짝마다 한 번 만들어 두고, `term_links`의 관계가 바뀐 짝만 다시 만든다.
- 기사 쪽 문장은 `news_company_reason.ipynb`(예상 가격 방향에 맞는 기업 뉴스 1문장)가 만든다. 화면은 두 문장을 이어 붙인다.
- 이 노트북은 DB의 `news.term_links`를 **읽기만** 한다. 프롬프트: `prompt/company_relation_prompt.md`(proj/prompt에서 복제).

## 1. 설정

- `STOCKS`: 만들 종목코드 목록. `None`이면 `term_links`의 모든 종목(해외 상대 직접 관계는 2026-11-20 기준 171종목 228짝).
- `KINDS`, `LINKS`: 대상 검색어 종류·연결. 기본은 `해외 상대` + `direct`(두 기업 사이 관계만).
- `BATCH_SIZE`: 한 번 호출에 넣을 짝 수. `RUN_LLM = False`이면 대상 짝과 LLM 입력만 보여 준다.
- `REBUILD = True`이면 이미 만든 짝도 다시 만든다.
- DB 주소는 `.env`의 `GNEWS_DB_URL`(읽기 전용 주소여도 된다), OpenAI 키는 `OPENAI_API_KEY`이며 값은 출력하지 않는다. `.env`는 노트북 폴더에서 위로 올라가며 처음 나오는 것을 쓴다.

In [ ]:
import datetime as dt
import hashlib
import json
import re
import urllib.error
import urllib.request
from pathlib import Path

import sqlalchemy as sa

import pandas as pd
from IPython.display import display

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더(이 노트북이 있는 폴더에서 연다)
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])   # .env가 있는 폴더(노트북 폴더에서 위로 올라가며 처음 나오는 곳)
RESULT_DIR = ROOT / 'source' / 'live'  # 다른 요약 노트북과 같은 결과 폴더
RELATION_PROMPT_PATH = ROOT / 'prompt' / 'company_relation_prompt.md'   # proj/prompt에서 복제
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI로 관계 문장을 만든다
STOCKS = None            # 만들 종목코드 목록. None이면 term_links의 모든 종목
KINDS = ['해외 상대']     # 대상 검색어 종류(두 기업 사이 관계)
LINKS = ['direct']       # 대상 연결(직접 관계)
BATCH_SIZE = 30          # 한 번 호출에 넣을 짝 수
REBUILD = False          # True면 이미 만든 짝도 다시 만든다
MAX_EVIDENCE_CHARS = 300 # 근거 문장은 이 길이까지만 넘긴다
# ==============================

DB_ENV = 'GNEWS_DB_URL'   # 서버 DB 주소(.env 이름). news.term_links를 읽는다
RELATIONS_PATH = RESULT_DIR / 'company_relations.jsonl'   # 짝마다 한 줄(다시 만들면 새 줄을 덧붙이고, 읽을 때 짝마다 마지막 줄을 쓴다)
USAGE_PATH = RESULT_DIR / 'llm_usage.csv'           # LLM 호출마다 토큰 수(다른 노트북과 함께 씀)
KST = dt.timezone(dt.timedelta(hours=9))
assert ROOT.name == 'news_search', 'news_search 폴더에서 열어야 한다.'

## 2. 작은 도구

`news_company_hot_issue.ipynb`와 같은 도구다.
- `env_value`: `.env`에서 키를 읽는다(값은 출력하지 않는다).
- `ask_openai`: OpenAI Responses API. 오류가 나면 오류 종류·코드만 알린다.

In [ ]:
def env_value(name):
    env_file = PROJ / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def read_jsonl(path):
    # 한 줄씩 읽는다. 다른 노트북이 쓰는 중인 마지막 줄(덜 쓰인 줄)은 건너뛴다.
    rows = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def append_jsonl(path, rows):
    with path.open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def append_csv(rows, path):
    if rows:
        data = pd.DataFrame(rows)
        if path.exists():   # 이미 있는 파일의 열 순서에 맞춘다
            data = data.reindex(columns=pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
        data.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


class LLMError(Exception):
    pass


def ask_openai(instructions, user_message, text_format):
    request = urllib.request.Request(
        'https://api.openai.com/v1/responses', method='POST',
        headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'},
        data=json.dumps({'model': LLM_MODEL, 'instructions': instructions, 'input': user_message, 'store': False,
                         'reasoning': {'effort': 'medium'}, 'text': {'format': text_format}}).encode('utf-8'))
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode('utf-8', 'replace')).get('error', {})
        except ValueError:
            err = {}
        raise LLMError(f"OpenAI HTTP {e.code} type={err.get('type')} code={err.get('code')}") from None
    text = ''.join(c.get('text', '') for item in body.get('output', []) if item.get('type') == 'message'
                   for c in item.get('content', []))
    return text, body.get('usage') or {}


def log_usage(step, ref, usage):
    append_csv([{'time_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'), 'step': step, 'ref': ref,
                 'input_tokens': usage.get('input_tokens'),
                 'cached_tokens': (usage.get('input_tokens_details') or {}).get('cached_tokens'),
                 'output_tokens': usage.get('output_tokens')}], USAGE_PATH)


print('OPENAI_API_KEY:', '설정됨' if env_value('OPENAI_API_KEY') else '미설정')

## 3. 대상 짝 고르기

1. DB `news.term_links`에서 `kind`가 `KINDS`, `link`가 `LINKS` 안인 줄을 읽는다(`STOCKS`가 있으면 그 종목만).
2. 같은 (종목코드, 상대 노드)의 줄(검색어만 다른 줄)을 하나로 합친다. 관계는 (관계 이름, 방향, 내용, 금액)이 같은 것을 한 번만 넣고, 근거 문장은 `MAX_EVIDENCE_CHARS`자까지 자른다.
3. 짝마다 관계 자료의 지문(`rel_hash`)을 만든다. `company_relations.jsonl`에 같은 짝·같은 지문이 이미 있으면 건너뛴다(`REBUILD = True`면 다시 만든다). `term_links`가 다시 만들어져 관계가 바뀐 짝만 새로 만든다.

In [ ]:
def load_pairs():
    """DB news.term_links → (종목, 상대 기업) 짝 목록(DataFrame)."""
    url = env_value(DB_ENV)
    if not url:
        raise RuntimeError(f'.env에 {DB_ENV}(서버 DB 주소)가 없다.')
    engine = sa.create_engine(url)
    try:
        with engine.connect() as con:
            tl = pd.read_sql(sa.text('select stock_code, company, kind, query, node, node_name, link, summary, relations, built_at '
                                     'from news.term_links where kind = any(:kinds) and link = any(:links)'),
                             con, params={'kinds': list(KINDS), 'links': list(LINKS)})
    finally:
        engine.dispose()
    tl['stock_code'] = tl['stock_code'].astype(str).str.zfill(6)
    if STOCKS is not None:
        tl = tl[tl['stock_code'].isin([str(c).zfill(6) for c in STOCKS])]
    pairs = []
    for (code, node), g in tl.groupby(['stock_code', 'node'], sort=False):
        rels, seen = [], set()
        for rel_list in g['relations']:
            for rel in (json.loads(rel_list) if isinstance(rel_list, str) else rel_list) or []:
                k = (rel.get('label'), rel.get('dir'), tuple(rel.get('notes') or []), tuple(rel.get('amounts') or []))
                if k in seen:
                    continue
                seen.add(k)
                rels.append({'label': rel.get('label'), 'dir': rel.get('dir'), 'notes': rel.get('notes') or [],
                             'amounts': rel.get('amounts') or [],
                             'evidence': [str(x)[:MAX_EVIDENCE_CHARS] for x in (rel.get('evidence') or [])[:2]]})
        rel_hash = hashlib.sha1(json.dumps(rels, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()[:16]
        pairs.append({'id': f'{code}|{node}', 'stock_code': code, 'company': g['company'].iloc[0], 'node': node,
                      'partner': g['node_name'].iloc[0], 'queries': list(dict.fromkeys(g['query'])), 'summary': g['summary'].iloc[0],
                      'relations': rels, 'rel_hash': rel_hash, 'term_links_built_at': str(g['built_at'].max())})
    return pd.DataFrame(pairs)


def todo_pairs():
    pairs = load_pairs()
    if pairs.empty:
        print('대상 짝이 없다.')
        return pairs
    done = {(r.get('id'), r.get('rel_hash')) for r in read_jsonl(RELATIONS_PATH)}
    todo = pairs if REBUILD else pairs[[(i, h) not in done for i, h in zip(pairs['id'], pairs['rel_hash'])]]
    print(f"대상 짝 {len(pairs)}개({pairs['stock_code'].nunique()}종목) 중 새로 만들 짝 {len(todo)}개"
          + ('' if REBUILD else f' (이미 만든 짝 {len(pairs) - len(todo)}개 건너뜀)'))
    return todo.reset_index(drop=True)

## 4. 관계 문장 만들기

- 짝을 `BATCH_SIZE`개씩 묶어 **한 번의 호출**로 OpenAI에 넘겨 `{"lines": [{"id", "line"}, ...]}`(짝마다 한 문장)을 받는다.
- 받은 문장을 `company_relations.jsonl`에 짝마다 한 줄씩 남긴다. 묶음마다 바로 저장한다(중간에 멈춰도 낸 비용이 남는다).
  - 칸: `id`(종목코드|상대 노드), `stock_code`, `company`, `node`, `partner`(상대 이름), `queries`(이 관계로 뉴스를 찾는 검색어), `summary`(term_links의 한 줄 요약), `rel_hash`, `term_links_built_at`, `created_kst`, `model`, `line`(관계 문장)
- 받은 `id`가 입력과 다르거나 빠지면 알리고, 그 짝은 저장하지 않는다(다음 실행 때 다시 만든다).

In [ ]:
RELATION_PROMPT = RELATION_PROMPT_PATH.read_text(encoding='utf-8')


def llm_pair(r):
    return {'id': r['id'], 'company': r['company'], 'partner': r['partner'], 'queries': r['queries'], 'relations': r['relations']}


def company_relations():
    todo = todo_pairs()
    if todo.empty:
        return {}
    display(todo[['id', 'company', 'partner', 'summary']])
    if not RUN_LLM:
        print('RUN_LLM = False라 OpenAI를 부르지 않는다. 첫 묶음의 LLM 입력:')
        print(json.dumps({'pairs': [llm_pair(r) for _, r in todo.head(BATCH_SIZE).iterrows()]}, ensure_ascii=False, indent=1))
        return {}
    made = {}
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo.iloc[i:i + BATCH_SIZE]
        text, usage = ask_openai(RELATION_PROMPT, 'pairs json:\n' + json.dumps({'pairs': [llm_pair(r) for _, r in batch.iterrows()]}, ensure_ascii=False),
                                 {'type': 'json_object'})
        log_usage('company_relation', f'{i // BATCH_SIZE + 1}:{len(batch)}', usage)
        got = {str(x.get('id', '')).strip(): str(x.get('line', '')).strip().lstrip('-•· ').strip()
               for x in json.loads(text).get('lines', []) if isinstance(x, dict)}
        wrong = sorted(set(got) - set(batch['id'])) + [p for p in batch['id'] if not got.get(p)]
        if wrong:
            print('주의: 입력과 맞지 않거나 빠진 짝(저장하지 않음):', wrong)
        now = dt.datetime.now(dt.timezone.utc).astimezone(KST).strftime('%Y-%m-%d %H:%M:%S')
        out = [{'id': r['id'], 'stock_code': r['stock_code'], 'company': r['company'], 'node': r['node'], 'partner': r['partner'],
                'queries': r['queries'], 'summary': r['summary'], 'rel_hash': r['rel_hash'], 'term_links_built_at': r['term_links_built_at'],
                'created_kst': now, 'model': LLM_MODEL, 'line': got[r['id']]}
               for _, r in batch.iterrows() if got.get(r['id'])]
        append_jsonl(RELATIONS_PATH, out)               # 묶음마다 바로 저장
        for r in out:
            print(f"{r['company']} – {r['partner']}: {r['line']}")
            made[r['id']] = r['line']
    return made

## 5. 실행

In [ ]:
relation_lines = company_relations()